In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import os, cv2, random, numpy as np


In [22]:
def corrupt_image(img):
    def add_noise(i):
        return np.clip(i + np.random.normal(0,20,i.shape),0,255).astype(np.uint8)
    def blur(i):
        return cv2.GaussianBlur(i,(5,5),0)
    def lowres(i):
        h,w=i.shape[:2]
        return cv2.resize(cv2.resize(i,(w//3,h//3)),(w,h))
    return random.choice([add_noise,blur,lowres])(img)


In [ ]:
class RAImageDataset(Dataset):
    def __init__(self, root_dir, img_size=224):
        self.paths=[os.path.join(root_dir,f) for f in os.listdir(root_dir)]

        self.transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((img_size,img_size))
        ])

    def __len__(self): return len(self.paths)

    def __getitem__(self,idx):
        img=cv2.imread(self.paths[idx])
        img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

        corrupted=corrupt_image(img.copy())

        return self.transform(corrupted), self.transform(img)


In [24]:
DATA_PATH = r"D:\Image Recognition\data"

dataset = RAImageDataset(DATA_PATH)
loader = DataLoader(dataset,batch_size=8,shuffle=True)
# print("Dataset:",len(dataset))


In [25]:
class Block(nn.Module):
    def __init__(self,a,b):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(a,b,3,1,1),
            nn.BatchNorm2d(b),
            nn.ReLU())
    def forward(self,x): return self.net(x)

class ProcessingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net=nn.Sequential(
            Block(3,64),Block(64,64),
            nn.MaxPool2d(2),
            Block(64,128),Block(128,128),
            nn.Upsample(scale_factor=2),
            Block(128,64),
            nn.Conv2d(64,3,3,1,1),
            nn.Sigmoid())
    def forward(self,x): return self.net(x)


In [26]:
device="cuda" if torch.cuda.is_available() else "cpu"
print("Using:",device)


Using: cuda


In [27]:
model=ProcessingNet().to(device)
model.load_state_dict(torch.load("baseline_model.pth", map_location=device))
print("Baseline weights loaded")


Baseline weights loaded


In [28]:
teacher=models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
teacher=teacher.to(device)
teacher.eval()

normalize = transforms.Normalize(
    mean=[0.485,0.456,0.406],
    std=[0.229,0.224,0.225]
)

print("ResNet50  ready")


ResNet50  ready


In [29]:
image_loss_fn=nn.MSELoss()
class_loss_fn=nn.CrossEntropyLoss()

lambda_recog=0.1
optimizer=optim.Adam(model.parameters(),lr=1e-4)


In [30]:
EPOCHS=10

for epoch in range(EPOCHS):

    total_loss=0
    pbar=tqdm(loader)

    for corrupted, clean in pbar:

        corrupted=corrupted.to(device)
        clean=clean.to(device)

        # restore
        restored=model(corrupted)

        # image loss
        img_loss=image_loss_fn(restored,clean)

        # teacher label
        with torch.no_grad():
            teacher_pred=teacher(normalize(clean))
            labels=teacher_pred.argmax(dim=1)

        # recognition loss
        pred=teacher(normalize(restored))
        recog_loss=class_loss_fn(pred,labels)

        loss=img_loss + lambda_recog*recog_loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        pbar.set_postfix(total=loss.item(),img=img_loss.item(),cls=recog_loss.item())

    print("Epoch",epoch+1,"Loss:",total_loss/len(loader))


100%|██████████| 375/375 [03:49<00:00,  1.64it/s, cls=1.46, img=0.00283, total=0.148]  


Epoch 1 Loss: 0.14648624128103258


100%|██████████| 375/375 [03:23<00:00,  1.84it/s, cls=1.2, img=0.00178, total=0.122]   


Epoch 2 Loss: 0.14056247893969218


100%|██████████| 375/375 [03:11<00:00,  1.95it/s, cls=1.58, img=0.00433, total=0.162]  


...

Epoch 8 Loss: 0.13433725225925447


100%|██████████| 375/375 [03:24<00:00,  1.83it/s, cls=1.24, img=0.00195, total=0.126]  


Epoch 9 Loss: 0.13272257312138874


100%|██████████| 375/375 [03:20<00:00,  1.87it/s, cls=1.15, img=0.00475, total=0.12]   

Epoch 10 Loss: 0.13340687495470047


In [31]:
torch.save(model.state_dict(),"RA_model_resnet50.pth")
print("RA ResNet50 model saved")


RA ResNet50 model saved
